# Прогнозирование спроса: EDA временных рядов

**Store Item Demand Forecasting — Task 1/3**

Цель этого ноутбука — подготовить панель из 500 ежедневных временных рядов и исследовать её структуру перед построением моделей прогнозирования.

**Что рассматривается:**
- проверка качества и регулярности данных;
- тренд и календарная сезонность;
- ACF/PACF и STL-декомпозиция;
- ADF unit-root диагностика;
- дифференцирование;
- различия масштаба между рядами.

> Результаты этого этапа определяют единую схему эксперимента для Task 2 и Task 3.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/time_series_final_project")

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / "train.csv"
TEST_PATH = RAW_DIR / "test.csv"
SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

print(TRAIN_PATH.exists(), TEST_PATH.exists(), SUBMISSION_PATH.exists())

## 1. Загрузка данных


### Набор данных

Датасет содержит **913 000 наблюдений** и **500 связанных временных рядов**: 10 магазинов × 50 товаров.  
Каждый ряд описывает ежедневные продажи конкретного товара в конкретном магазине.

- **Период:** 2013-01-01 — 2017-12-31
- **Частота:** 1 день
- **Единица ряда:** `store × item`
- **Целевая переменная:** `sales`


### Постановка задачи

Предметная область — **прогнозирование спроса в розничной торговле**.  
Данные образуют панель временных рядов, поэтому важно анализировать как общую динамику продаж, так и различия между отдельными парами `store-item`.

В дальнейшем исходные данные приводятся к единому формату:

`unique_id | ds | y | store | item`


### Дизайн прогнозирования

Для следующих этапов проекта фиксируется единая постановка:

| Параметр | Значение |
|---|---|
| Горизонт прогноза | **28 дней** |
| Режим | **offline forecasting** |
| Основная относительная метрика | **sMAPE** |
| Дополнительные метрики | **MAE, RMSE** |
| Подходы | статистические модели, глобальные ML/DL-модели |

Статистические модели оцениваются отдельно для каждого ряда, а глобальные ML/DL-модели обучаются сразу на множестве рядов. Наличие 500 серий позволяет сравнить оба подхода в одинаковой схеме backtesting.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

df_raw = pd.read_csv(TRAIN_PATH)

display(df_raw.head())
print("Размер train.csv:", df_raw.shape)

### Структура исходного датасета


In [ ]:
df_raw.info()

### Описательная статистика


In [ ]:
display(df_raw.describe(include="all"))

### Проверка качества данных

Проверяем пропуски, полные дубликаты и далее — уникальность ключа `unique_id + ds`.


In [ ]:
print("Пропуски по колонкам:")
display(df_raw.isna().sum())

print("Количество полных дубликатов строк:", df_raw.duplicated().sum())
print(
    "Количество дубликатов бизнес-ключа (date, store, item):",
    df_raw.duplicated(["date", "store", "item"]).sum(),
)

## 2. Подготовка временных рядов

Для `StatsForecast`, `MLForecast` и `NeuralForecast` используется длинный формат:

- `unique_id` — идентификатор ряда (`store + item`);
- `ds` — дата;
- `y` — целевая переменная (`sales`);
- `store`, `item` — статические признаки ряда.


In [ ]:
df = df_raw.copy()

df["date"] = pd.to_datetime(df["date"])

df["unique_id"] = "store_" + df["store"].astype(str) + "_item_" + df["item"].astype(str)

df = df.rename(columns={"date": "ds", "sales": "y"})

df = df[["unique_id", "ds", "y", "store", "item"]]
df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

display(df.head())
display(df.tail())

In [ ]:
print(df.dtypes)

### Размерность панели и временной диапазон


In [ ]:
print("Количество временных рядов:", df["unique_id"].nunique())
print("Количество магазинов:", df["store"].nunique())
print("Количество товаров:", df["item"].nunique())

print("Минимальная дата:", df["ds"].min())
print("Максимальная дата:", df["ds"].max())
print("Количество наблюдений:", len(df))

series_lengths = df.groupby("unique_id").size()

print("Минимальная длина ряда:", series_lengths.min())
print("Максимальная длина ряда:", series_lengths.max())
print("Средняя длина ряда:", series_lengths.mean())

### Регулярность временной сетки

Для каждого `unique_id` проверяем отсутствие повторяющихся дат и пропущенных календарных дней.


In [ ]:
# Проверяем не только количество строк, но и уникальность дат внутри каждого ряда.
duplicate_keys = df.duplicated(["unique_id", "ds"]).sum()

# Число уникальных дат и ожидаемое число календарных дней между min и max.
date_check = (
    df.groupby("unique_id")["ds"]
    .agg(min_date="min", max_date="max", n_unique_dates="nunique", n_rows="size")
    .reset_index()
)
date_check["expected_days"] = (
    date_check["max_date"] - date_check["min_date"]
).dt.days + 1
date_check["missing_dates"] = date_check["expected_days"] - date_check["n_unique_dates"]
date_check["extra_rows"] = date_check["n_rows"] - date_check["n_unique_dates"]

print("Дубликатов (unique_id, ds):", duplicate_keys)
print(
    "Рядов с пропущенными календарными датами:", (date_check["missing_dates"] > 0).sum()
)
print("Рядов с повторяющимися датами:", (date_check["extra_rows"] > 0).sum())
display(date_check.head())

assert duplicate_keys == 0, "Есть дубликаты ключа (unique_id, ds)"
assert (date_check["missing_dates"] == 0).all(), "Есть пропуски дат"
assert (date_check["extra_rows"] == 0).all(), "Есть повторяющиеся даты"

### Сохранение подготовленного датасета

Подготовленная таблица используется без повторной обработки в Task 2 и Task 3.


In [ ]:
PREPARED_PATH = PROCESSED_DIR / "store_item_sales_prepared.csv"

df.to_csv(PREPARED_PATH, index=False)

print("Сохранено:", PREPARED_PATH)

## 3. Разведочный анализ данных


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

### Общая динамика продаж

Агрегируем продажи по календарной дате, чтобы оценить общий тренд и крупномасштабные изменения спроса.


In [ ]:
daily_sales = (
    df.groupby("ds", as_index=False)["y"].sum().rename(columns={"y": "total_sales"})
)

display(daily_sales.head())

plt.figure(figsize=(16, 5))
plt.plot(daily_sales["ds"], daily_sales["total_sales"])
plt.title("Общий объём продаж по дням")
plt.xlabel("Дата")
plt.ylabel("Суммарные продажи")
plt.show()

#### Сохранение графика

Визуализация сохраняется отдельно, чтобы её можно было использовать в отчёте или README.


In [ ]:
fig_path = FIGURES_DIR / "total_daily_sales.png"

plt.figure(figsize=(16, 5))
plt.plot(daily_sales["ds"], daily_sales["total_sales"])
plt.title("Общий объём продаж по дням")
plt.xlabel("Дата")
plt.ylabel("Суммарные продажи")
plt.grid(True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Сохранено:", fig_path)

### Динамика продаж по годам


In [ ]:
df_eda = df.copy()
df_eda["year"] = df_eda["ds"].dt.year
df_eda["month"] = df_eda["ds"].dt.month
df_eda["dayofweek"] = df_eda["ds"].dt.dayofweek
df_eda["day_name"] = df_eda["ds"].dt.day_name()

year_sales = (
    df_eda.groupby("year", as_index=False)["y"]
    .sum()
    .rename(columns={"y": "total_sales"})
)

display(year_sales)

plt.figure(figsize=(10, 5))
plt.bar(year_sales["year"], year_sales["total_sales"])
plt.title("Суммарные продажи по годам")
plt.xlabel("Год")
plt.ylabel("Суммарные продажи")
plt.show()

### Сезонность по месяцам


In [ ]:
month_sales = (
    df_eda.groupby("month", as_index=False)["y"]
    .mean()
    .rename(columns={"y": "avg_sales"})
)

display(month_sales)

plt.figure(figsize=(10, 5))
plt.plot(month_sales["month"], month_sales["avg_sales"], marker="o")
plt.title("Средние продажи по месяцам")
plt.xlabel("Месяц")
plt.ylabel("Средние продажи")
plt.xticks(range(1, 13))
plt.show()

### Сезонность по дням недели


In [ ]:
dow_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

dow_sales = (
    df_eda.groupby("day_name", as_index=False)["y"]
    .mean()
    .rename(columns={"y": "avg_sales"})
)

dow_sales["day_name"] = pd.Categorical(
    dow_sales["day_name"], categories=dow_order, ordered=True
)
dow_sales = dow_sales.sort_values("day_name")

display(dow_sales)

plt.figure(figsize=(10, 5))
plt.bar(dow_sales["day_name"].astype(str), dow_sales["avg_sales"])
plt.title("Средние продажи по дням недели")
plt.xlabel("День недели")
plt.ylabel("Средние продажи")
plt.xticks(rotation=45)
plt.show()

### Различия между магазинами


In [ ]:
store_sales = (
    df_eda.groupby("store", as_index=False)["y"]
    .sum()
    .rename(columns={"y": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

display(store_sales)

plt.figure(figsize=(10, 5))
plt.bar(store_sales["store"].astype(str), store_sales["total_sales"])
plt.title("Суммарные продажи по магазинам")
plt.xlabel("Магазин")
plt.ylabel("Суммарные продажи")
plt.show()

### Различия между товарами


In [ ]:
item_sales = (
    df_eda.groupby("item", as_index=False)["y"]
    .sum()
    .rename(columns={"y": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

display(item_sales.head(10))

plt.figure(figsize=(14, 5))
plt.bar(item_sales["item"].astype(str), item_sales["total_sales"])
plt.title("Суммарные продажи по товарам")
plt.xlabel("Товар")
plt.ylabel("Суммарные продажи")
plt.show()

### Пример отдельного временного ряда

Для детального анализа далее используется `store_1_item_1`.


In [ ]:
one_id = "store_1_item_1"

one_series = df[df["unique_id"] == one_id].copy()

display(one_series.head())
display(one_series.tail())

plt.figure(figsize=(16, 5))
plt.plot(one_series["ds"], one_series["y"])
plt.title(f"Временной ряд продаж: {one_id}")
plt.xlabel("Дата")
plt.ylabel("Продажи")
plt.show()

### Скользящие средние

Сравниваем исходный ряд с 7-дневным и 30-дневным rolling mean, чтобы визуально отделить краткосрочные колебания от более медленной динамики.


In [ ]:
one_series["rolling_7"] = one_series["y"].rolling(window=7).mean()
one_series["rolling_30"] = one_series["y"].rolling(window=30).mean()

plt.figure(figsize=(16, 5))
plt.plot(one_series["ds"], one_series["y"], label="actual", alpha=0.5)
plt.plot(one_series["ds"], one_series["rolling_7"], label="rolling 7")
plt.plot(one_series["ds"], one_series["rolling_30"], label="rolling 30")
plt.title(f"Сглаживание временного ряда: {one_id}")
plt.xlabel("Дата")
plt.ylabel("Продажи")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["y"], bins=50)
plt.title("Распределение продаж")
plt.xlabel("Продажи")
plt.ylabel("Количество наблюдений")
plt.grid(True)
plt.show()

display(df["y"].describe())

#### Распределение продаж

Общее распределение `y` правосторонне асимметрично, однако оно объединяет 500 рядов с разными уровнями спроса. Поэтому длинный правый хвост нельзя автоматически трактовать как набор аномалий: часть высоких значений обусловлена нормальной неоднородностью магазинов и товаров.

Потенциальные выбросы корректнее анализировать **внутри конкретного временного ряда** и с учётом его локального уровня и сезонности.


In [ ]:
plt.figure(figsize=(12, 5))
df_eda.boxplot(column="y", by="store")
plt.title("Распределение продаж по магазинам")
plt.suptitle("")
plt.xlabel("Магазин")
plt.ylabel("Продажи")
plt.show()

#### Boxplot по магазинам

Магазины различаются и по среднему уровню, и по разбросу продаж. Точки за пределами «усов» boxplot являются статистически экстремальными по правилу IQR, но не обязательно ошибками данных или бизнес-аномалиями.

Для вывода об аномальности необходимо учитывать конкретный `store-item` и положение наблюдения во времени.


In [ ]:
zero_sales = (
    df.groupby("unique_id")
    .agg(
        n_obs=("y", "size"),
        zero_count=("y", lambda x: (x == 0).sum()),
        min_sales=("y", "min"),
        mean_sales=("y", "mean"),
        max_sales=("y", "max"),
    )
    .reset_index()
)

zero_sales["zero_share"] = zero_sales["zero_count"] / zero_sales["n_obs"]

display(zero_sales.sort_values("zero_share", ascending=False).head(20))
display(zero_sales["zero_share"].describe())

print("Рядов с хотя бы одним нулём:", (zero_sales["zero_count"] > 0).sum())
print("Максимальная доля нулей в одном ряду:", zero_sales["zero_share"].max())

### Нулевые продажи

Для каждого `unique_id` рассчитана доля наблюдений с `y = 0`. Только один ряд — `store_6_item_4` — содержит одно нулевое значение из 1826 наблюдений; максимальная доля нулей составляет **0.0548%**.

Следовательно, выраженной проблемы **intermittent demand** в датасете нет. Это позволяет использовать стандартные методы прогнозирования без отдельной обработки большого количества нулевых наблюдений.


In [ ]:
fig_path = FIGURES_DIR / "zero_sales_share_distribution.png"

plt.figure(figsize=(10, 5))
plt.hist(zero_sales["zero_share"], bins=30)
plt.title("Распределение доли нулевых продаж по временным рядам")
plt.xlabel("Доля нулевых продаж")
plt.ylabel("Количество рядов")
plt.grid(True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Сохранено:", fig_path)

### Промежуточные выводы EDA

1. На агрегированном уровне виден долгосрочный рост продаж.
2. Наблюдается выраженная календарная структура, включая недельную сезонность.
3. Ряды существенно различаются по масштабу, поэтому агрегированное распределение смешивает неоднородные группы.
4. Нулевые продажи практически отсутствуют — задача не относится к типичному intermittent-demand forecasting.
5. Временную зависимость и детерминированные компоненты следует дополнительно проверить через ACF/PACF, STL и unit-root диагностику.


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, acf

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## 4. Детальный анализ одного ряда


In [ ]:
one_id = "store_1_item_1"

one_series = df[df["unique_id"] == one_id].copy().sort_values("ds")

one_series = one_series.set_index("ds")

display(one_series.head())
display(one_series.tail())

y = one_series["y"]

Для диагностики выбран ряд **`store_1_item_1`** — продажи товара 1 в магазине 1. На нём анализируются автокорреляция, сезонность, декомпозиция и чувствительность ADF-теста к спецификации детерминированной части.


### ACF и PACF

Графики ACF/PACF используются для оценки зависимости между текущими значениями ряда и его лагами. Особое внимание уделяется лагам, кратным 7, поскольку данные имеют дневную частоту.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(y, lags=60, ax=axes[0])
axes[0].set_title(f"ACF для ряда {one_id}")
axes[0].set_xlabel("Лаг")
axes[0].set_ylabel("Автокорреляция")

plot_pacf(y, lags=60, ax=axes[1], method="ywm")
axes[1].set_title(f"PACF для ряда {one_id}")
axes[1].set_xlabel("Лаг")
axes[1].set_ylabel("Частичная автокорреляция")
plt.tight_layout()
plt.show()

acf_values = acf(y, nlags=60, fft=True)
print(f"ACF на лаге 7 для {one_id}: {acf_values[7]:.3f}")
print("Пики на лагах, кратных 7, поддерживают гипотезу о недельной сезонности.")

### STL-декомпозиция


Сначала рассматриваем **недельную сезонность** с `period=7`, естественную для ежедневных данных.


In [ ]:
stl_weekly = STL(y, period=7)
stl_weekly_result = stl_weekly.fit()

fig = stl_weekly_result.plot()
fig.set_size_inches(14, 8)
plt.suptitle(f"STL-декомпозиция ряда {one_id}, period=7", y=1.02)
plt.savefig(FIGURES_DIR / "stl_weekly_store_1_item_1.png", dpi=150, bbox_inches="tight")
plt.show()

Дополнительно выполняется STL с `period=365` как визуальная диагностика более медленного годового календарного паттерна. Эта декомпозиция используется для анализа структуры, а не как прямое указание на конкретную SARIMA-спецификацию.


In [ ]:
stl_yearly = STL(y, period=365)
stl_yearly_result = stl_yearly.fit()

fig = stl_yearly_result.plot()
fig.set_size_inches(14, 8)
plt.suptitle(f"STL-декомпозиция ряда {one_id}, period=365", y=1.02)
plt.savefig(FIGURES_DIR / "stl_yearly_store_1_item_1.png", dpi=150, bbox_inches="tight")
plt.show()

### ADF-тест исходного ряда


In [ ]:
def run_adf_test(series, name, regression="c"):
    s = pd.Series(series).dropna()
    result = adfuller(s, regression=regression, autolag="AIC", result_object=False)
    return {
        "series": name,
        "regression": regression,
        "adf_statistic": result[0],
        "p_value": result[1],
        "n_lags": result[2],
        "n_obs": result[3],
        "reject_unit_root_5pct": result[1] < 0.05,
    }

In [ ]:
adf_original_c = run_adf_test(y, "original", regression="c")
adf_original_ct = run_adf_test(y, "original", regression="ct")
adf_original_df = pd.DataFrame([adf_original_c, adf_original_ct])
display(adf_original_df)

print(
    "Важно: если вывод меняется между 'c' и 'ct', нельзя сводить результат к фразе "
    "«ряд стационарен/нестационарен» без указания спецификации теста."
)

#### Как интерпретировать ADF

ADF-тест проверяет нулевую гипотезу о наличии **единичного корня**. Он не является универсальным тестом на «отсутствие тренда или сезонности».

Результат зависит от включённых детерминированных компонент. Поэтому сравниваются две спецификации:

- `regression="c"` — константа;
- `regression="ct"` — константа + линейный тренд.


Для `store_1_item_1` результаты различаются: при `regression="c"` нулевая гипотеза отвергается на уровне 5%, а при `regression="ct"` — нет.

Это означает, что вывод **чувствителен к спецификации теста**. Поэтому здесь не присваивается упрощённый ярлык «стационарный/нестационарный» только по одному p-value; структура ряда оценивается совместно с графиками, ACF и STL.


In [ ]:
# ADF по всем 500 рядам в двух спецификациях: c и ct.
adf_results = []

for uid, g in df.groupby("unique_id", sort=False):
    s = g.sort_values("ds")["y"]
    for regression in ("c", "ct"):
        try:
            r = run_adf_test(s, uid, regression=regression)
            adf_results.append(r)
        except Exception as e:
            adf_results.append(
                {
                    "series": uid,
                    "regression": regression,
                    "adf_statistic": np.nan,
                    "p_value": np.nan,
                    "n_lags": np.nan,
                    "n_obs": len(s),
                    "reject_unit_root_5pct": False,
                    "error": str(e),
                }
            )

adf_all = pd.DataFrame(adf_results)
summary_adf = (
    adf_all.groupby("regression")
    .agg(
        n_series=("series", "nunique"),
        n_reject_5pct=("reject_unit_root_5pct", "sum"),
        median_p=("p_value", "median"),
    )
    .reset_index()
)
summary_adf["share_reject_5pct"] = (
    summary_adf["n_reject_5pct"] / summary_adf["n_series"]
)
display(summary_adf)

# Сохраняем полную таблицу для воспроизводимости.
adf_all.to_csv(PROCESSED_DIR / "adf_results_all_series.csv", index=False)

#### ADF по всем 500 рядам

При спецификации `c` нулевая гипотеза отвергается для **86.8%** рядов, а при `ct` — только для **5.8%**. Такое расхождение показывает, насколько сильно вывод зависит от учёта детерминированного тренда.

Поэтому корректнее говорить не «86.8% рядов стационарны», а: **результат unit-root диагностики неустойчив к спецификации детерминированной части**. ADF здесь используется как один из диагностических инструментов, а порядок дифференцирования в ARIMA-подходах должен подтверждаться модельным отбором и out-of-sample качеством.


### Дифференцирование


Проверяются два преобразования:

1. первое дифференцирование: `y[t] - y[t-1]`;
2. недельное сезонное дифференцирование: `y[t] - y[t-7]`.

Первое преобразование ориентировано на медленную динамику уровня, второе — на повторяющийся недельный паттерн.


In [ ]:
one_series["diff_1"] = one_series["y"].diff(1)
one_series["diff_7"] = one_series["y"].diff(7)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(one_series.index, one_series["diff_1"])
ax.set_title(f"Первое дифференцирование ряда {one_id}")
ax.set_xlabel("Дата")
ax.set_ylabel("y[t] - y[t-1]")
plt.grid(True)
plt.savefig(FIGURES_DIR / "diff_1_store_1_item_1.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(one_series.index, one_series["diff_7"])
ax.set_title(f"Сезонное дифференцирование ряда {one_id}, lag=7")
ax.set_xlabel("Дата")
ax.set_ylabel("y[t] - y[t-7]")
plt.grid(True)
plt.savefig(FIGURES_DIR / "diff_7_store_1_item_1.png", dpi=150, bbox_inches="tight")
plt.show()

#### Зачем проверять дифференцирование

Для исходного ряда результаты ADF зависят от спецификации (`c` против `ct`), а визуальный анализ и STL показывают выраженные медленные и сезонные компоненты. Поэтому дифференцирование рассматривается **как диагностическое и модельное преобразование**, а не как следствие одного теста.

Недельное `diff(7)` особенно важно проверить из-за устойчивой сезонности на лагах, кратных 7. При этом окончательная необходимость и порядок дифференцирования должны оцениваться внутри конкретной модели и по out-of-sample backtesting.


#### ADF после дифференцирования


In [ ]:
adf_diff_results = []
for name, series in [
    ("first_difference", one_series["diff_1"]),
    ("seasonal_difference_7", one_series["diff_7"]),
]:
    for regression in ("c", "ct"):
        adf_diff_results.append(run_adf_test(series, name, regression=regression))

adf_results_diff = pd.DataFrame(adf_diff_results)
display(adf_results_diff)

После `diff(1)` и `diff(7)` ADF отвергает нулевую гипотезу о единичном корне в обеих спецификациях (`c` и `ct`) с очень малыми p-value.

Это является сильным свидетельством против unit-root поведения после преобразования, но **не доказывает полную стационарность во всех смыслах**. Сезонность, изменение дисперсии и остаточная автокорреляция должны проверяться отдельно.


#### Автоматическая сводка ADF


In [ ]:
for _, row in adf_results_diff.iterrows():
    conclusion = "reject H0" if row["reject_unit_root_5pct"] else "fail to reject H0"
    print(
        f"{row['series']}, regression={row['regression']}: "
        f"p-value={row['p_value']:.5g} -> {conclusion}"
    )

In [ ]:
print("Итоговая интерпретация:")
print("1) ADF проверяет unit root, а не отсутствие сезонности как таковой.")
print("2) Для исходного ряда вывод чувствителен к включению детерминированного тренда.")
print(
    "3) Первое и сезонное дифференцирование рассматриваются как диагностические/модельные преобразования,"
)
print(
    "   но их необходимость должна подтверждаться конкретной моделью и out-of-sample валидацией."
)

## 5. Масштаб временных рядов


In [ ]:
# Анализ масштаба временных рядов

scale_stats = (
    df.groupby("unique_id")["y"]
    .agg(["min", "mean", "std", "max", "median"])
    .reset_index()
)

display(scale_stats.head())
display(scale_stats[["min", "mean", "std", "max", "median"]].describe())

scale_stats.to_csv(PROCESSED_DIR / "scale_stats_by_unique_id.csv", index=False)

Средний спрос по отдельным `store-item` варьируется примерно от **13 до 113 единиц в день**, а стандартное отклонение — от **4.7 до 29.7**. Максимальные значения по сериям находятся в диапазоне от 31 до 231.

Такой разброс особенно важен для глобальных ML/DL-моделей: без локального масштабирования крупные ряды могут сильнее влиять на абсолютные функции потерь. Поэтому в Task 3 используется масштабирование target отдельно по каждому `unique_id`.


### Распределение среднего спроса по рядам


In [ ]:
fig_path = FIGURES_DIR / "mean_sales_by_series_distribution.png"

plt.figure(figsize=(10, 5))
plt.hist(scale_stats["mean"], bins=30)
plt.title("Распределение средних продаж по временным рядам")
plt.xlabel("Среднее значение продаж")
plt.ylabel("Количество рядов")
plt.grid(True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Сохранено:", fig_path)

### Распределение стандартного отклонения


In [ ]:
fig_path = FIGURES_DIR / "std_sales_by_series_distribution.png"

plt.figure(figsize=(10, 5))
plt.hist(scale_stats["std"], bins=30)
plt.title("Распределение стандартного отклонения продаж по временным рядам")
plt.xlabel("Стандартное отклонение продаж")
plt.ylabel("Количество рядов")
plt.grid(True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Сохранено:", fig_path)

### Средний спрос по матрице `store × item`


In [ ]:
# Heatmap средних продаж store × item

import seaborn as sns

cross_sales = df_eda.pivot_table(
    index="store", columns="item", values="y", aggfunc="mean"
)

display(cross_sales.head())

fig_path = FIGURES_DIR / "store_item_mean_sales_heatmap.png"

plt.figure(figsize=(16, 7))
sns.heatmap(cross_sales, cmap="viridis")
plt.title("Средние продажи по парам магазин–товар")
plt.xlabel("Товар")
plt.ylabel("Магазин")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Сохранено:", fig_path)

Heatmap показывает, что уровень спроса зависит и от магазина, и от товара. Некоторые товары имеют стабильно высокий спрос в разных магазинах, а сами магазины различаются по общему масштабу продаж.

Это подтверждает корректность пары `store-item` как единицы временного ряда и мотивирует использование `store` и `item` в качестве статических признаков глобальных моделей.


## 6. Итоговые выводы

- Датасет содержит **500 регулярных ежедневных рядов** без пропусков временной сетки и дубликатов ключа.
- Наблюдаются долгосрочная динамика и выраженная **недельная сезонность**; также присутствует более медленный календарный паттерн.
- Нулевые продажи единичны, поэтому отдельная стратегия для intermittent demand не требуется.
- Серии заметно отличаются по уровню и дисперсии — это важно учитывать при обучении глобальных моделей.
- ADF-диагностика чувствительна к спецификации детерминированной части; её нельзя интерпретировать отдельно от графического анализа и модельной валидации.
- Для следующих этапов фиксируются: `horizon=28`, `season_length=7`, rolling-origin backtesting и сравнение моделей по **sMAPE**, **MAE** и **RMSE**.

Следующий этап — сравнение статистических моделей на единой схеме backtesting.
